In [ ]:
from pathlib import Path
from IPython.display import Image, SVG, display
import getpass
import os
import re
import time

# Works when this notebook starts in notebooks/, the repository root, or Gadi.
candidates = [Path.cwd(), *Path.cwd().parents,
              Path(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))]
repo_root = next((path for path in candidates
                  if (path / "exercises/CMakeLists.txt").is_file()
                  and (path / "job-script").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the intro-to-cuda checkout.")
os.chdir(repo_root)
job_dir = repo_root / "job-script"
username = getpass.getuser()


def show_job_output(job_name, job_id):
    job_number = job_id.split(".")[0]
    log = job_dir / f"{job_name}.o{job_number}"
    if log.is_file():
        print(log.read_text())
    else:
        print(f"Output is not available yet: {log.name}. Check qstat, then rerun this cell.")


def outputs_ready(paths, submitted_at):
    if all(path.is_file() and path.stat().st_mtime >= submitted_at for path in paths):
        return True
    print("New results are not available yet. Wait for this job to finish, then rerun this cell.")
    return False

print(f"Repository: {repo_root}")

# Asynchronous CUDA Calls


> **Overview**
>
> 
>  * **Time:** 60 min
> 
>  1. Learn how to use asynchronous CUDA APIs to improve performance.
>  1. Understand the concept of CUDA streams and how they can be used to overlap memory transfers and kernel execution.

This section provides an example of how to use asynchronous CUDA APIs to improve the performance of a simple vector addition program.
By using CUDA streams, we can overlap memory transfers and kernel execution, leading to better GPU utilization.

## CUDA Streams


A stream is a sequence of operations that execute on the GPU in the order they are issued. Streams allow for concurrent execution of multiple operations, such
as memory transfers and kernel launches, which can significantly improve performance by overlapping computation and data transfer.

In [ ]:
# Streams Overview
display(Image(filename=str(repo_root / "docs/source/tutorial/figs/streams.png")))

Streams are particularly useful in scenarios where you want to perform multiple operations on the GPU without waiting for each operation to complete before
starting the next one. By using streams, you can maximize the utilization of the GPU and reduce idle time.

## Why streams result in more concurrency?


1. Streaming Multiprocessors (SMs)

    * GPUs contain multiple Streaming Multiprocessors (SMs).
    * Each SM can execute threads from different kernels concurrently, as long as there's enough space and resources (registers, shared memory, etc.).
    * So parallel kernel launches can utilize multiple SMs, allowing for more threads to be executed simultaneously.

2. Separate Engines for Compute and Memory Operations

    * GPUs have separate engines for memory operations (like `cudaMemcpyAsync`) and compute operations (like kernel launches).
    * This means that while one stream is executing a kernel, another stream can perform memory transfers without waiting for the first to finish.
    * This overlap allows for better utilization of the GPU's resources.


For efficient and proper use of streams, it's important to ensure that

* The GPU must support concurrent kernel execution.
* There must be no resource conflict (e.g., using different memory regions).
* Streams must be independent — no implicit synchronization (e.g., from using the default stream).


> **Important**
>
> Streams are not a replacement for parallelism; they are a way to manage concurrency on the GPU. You can have multiple streams executing concurrently, 
> but each stream's operations will still be executed in order.

### How two streams overlap

Follow the moving cursor. Each stream copies its inputs to the GPU, runs vector
addition, then copies the result back. At time 3, stream 0 is computing while
stream 1 is copying inputs. At time 6, stream 1 is computing while stream 0 is
copying its result back.

This timeline illustrates possible overlap. Actual overlap depends on the
device and workload. Run the cell on its own without CUDA or PBS.


In [ ]:
from IPython.display import HTML, display

# Each tuple holds an operation's label and colour.
operations = [
    ("Copy to GPU", "#1d4ed8"),
    ("Vector add", "#6d28d9"),
    ("Copy to CPU", "#047857"),
]
stream_boundaries = [(0, 2, 5, 7), (2, 5, 8, 10)]
left, step = 96, 50
blocks = []
for stream, boundaries in enumerate(stream_boundaries):
    y = 25 + stream * 52
    blocks.append(f'<text x="8" y="{y + 21}">Stream {stream}</text>')
    for (label, color), start, end in zip(operations, boundaries, boundaries[1:]):
        x, width = left + step * start, step * (end - start)
        blocks.append(
            f'<rect x="{x}" y="{y}" width="{width}" height="34" rx="4" fill="{color}"/>'
            f'<text x="{x + width / 2}" y="{y + 22}" text-anchor="middle" fill="white">{label}</text>'
        )

ticks = []
for time in range(0, 11, 2):
    x = left + step * time
    ticks.append(f'<text x="{x}" y="139" text-anchor="middle">{time}</text>')

display(HTML("""
<style>
.cuda-stream-demo { max-width: 620px; }
.cuda-stream-demo svg { display: block; width: 100%; height: auto; }
.cuda-stream-demo .time-cursor { animation: cuda-stream-sweep 8s linear infinite; }
.cuda-stream-demo input:checked ~ svg .time-cursor { animation-play-state: paused; }
@keyframes cuda-stream-sweep { to { transform: translateX(500px); } }
@media (prefers-reduced-motion: reduce) {
    .cuda-stream-demo .time-cursor { display: none; }
}
</style>
<div class="cuda-stream-demo">
<input type="checkbox" id="cuda-stream-pause">
<label for="cuda-stream-pause">Pause cursor</label>
<svg viewBox="0 0 620 169" role="img" aria-label="Two CUDA streams with overlapping copies and vector addition">
  <rect width="620" height="169" rx="8" fill="#f8fafc"/>
  <g font-family="system-ui, sans-serif" font-size="12" fill="#172033">
""" + "".join(blocks) + "".join(ticks) + """
    <path d="M96 121 H596" stroke="#64748b"/>
    <text x="346" y="159" text-anchor="middle">Illustrative time</text>
  </g>
  <g class="time-cursor" aria-hidden="true">
    <path d="M96 17 V121" stroke="white" stroke-width="4"/>
    <path d="M96 17 V121" stroke="#111827" stroke-width="2" stroke-dasharray="4 3"/>
  </g>
</svg>
</div>
"""))


### The host keeps going after an async call returns

Follow the cursor across the two lanes. The host queues a copy to the GPU,
a kernel launch, and a copy back. Each call returns after submission in this
example. The operations execute in order on one GPU stream, while the host
continues with independent CPU work.

This example assumes pinned host buffers for the copies and omits
`cudaStreamSynchronize()`. When the host's last enqueue call returns, the
kernel is still running. The result is ready for the host only after the
copy back completes. The timeline is illustrative and runs without CUDA or PBS.


In [ ]:
from IPython.display import HTML, display

# Illustrative intervals on one shared timeline, in SVG coordinates.
host = [
    (90, 180, "Queue copy in", "#1d4ed8"),
    (180, 270, "Launch kernel", "#6d28d9"),
    (270, 360, "Queue copy out", "#047857"),
    (360, 650, "Continue CPU work", "#334155"),
]
gpu = [
    (140, 275, "Copy to GPU", "#1d4ed8"),
    (275, 510, "Run kernel", "#6d28d9"),
    (510, 630, "Copy to CPU", "#047857"),
]
bars = []
for y, operations in [(48, host), (105, gpu)]:
    for start, end, label, color in operations:
        bars.append(
            f'<rect x="{start}" y="{y}" width="{end - start - 2}" height="32" rx="4" fill="{color}"/>'
            f'<text x="{(start + end) / 2}" y="{y + 20}" text-anchor="middle" fill="white">{label}</text>'
        )

display(HTML("""
<style>
.cuda-async-return { max-width: 675px; }
.cuda-async-return svg { display: block; width: 100%; height: auto; }
.cuda-async-return .queue-cursor { animation: cuda-async-return-sweep 9s linear infinite; }
.cuda-async-return input:checked ~ svg .queue-cursor { animation-play-state: paused; }
@keyframes cuda-async-return-sweep { to { transform: translateX(560px); } }
@media (prefers-reduced-motion: reduce) {
    .cuda-async-return .queue-cursor { display: none; }
}
</style>
<div class="cuda-async-return">
<input type="checkbox" id="cuda-async-return-pause">
<label for="cuda-async-return-pause">Pause cursor</label>
<svg viewBox="0 0 675 218" role="img" aria-labelledby="cuda-async-return-title">
  <title id="cuda-async-return-title">The host continues before the GPU finishes its queued work</title>
  <rect width="675" height="218" rx="8" fill="#f8fafc"/>
  <g font-family="system-ui, sans-serif" font-size="11" fill="#172033">
    <text x="90" y="28" font-size="15" font-weight="700">One stream, no host wait</text>
    <text x="14" y="68" font-weight="700">Host (CPU)</text>
    <text x="14" y="125" font-weight="700">GPU stream</text>
""" + "".join(bars) + """
    <path d="M360 83 V101 M360 141 V156 M630 141 V156" stroke="#64748b" stroke-dasharray="3 3"/>
    <text x="360" y="173" text-anchor="middle">Last enqueue call returns</text>
    <text x="630" y="173" text-anchor="end">Copy out completes</text>
    <path d="M90 190 H650 l-6 -4 m6 4 l-6 4" fill="none" stroke="#64748b"/>
    <text x="90" y="207">Illustrative time</text>
  </g>
  <g class="queue-cursor" aria-hidden="true">
    <path d="M90 39 V148" stroke="white" stroke-width="4"/>
    <path d="M90 39 V148" stroke="#111827" stroke-width="2" stroke-dasharray="4 3"/>
  </g>
</svg>
</div>
"""))


### How streams keep different GPU engines busy

Chunks A, B, and C each use their own stream. Within a chunk, the input copy
finishes before its kernel runs, and the output copy follows the kernel.
In the highlighted interval, one copy engine brings in C, the SMs run B's
kernel, and another copy engine sends A's result back to the host.

With equal-length steps, this example completes in five time slots instead of
nine. The timeline assumes support for simultaneous copies in both directions
and kernel execution, pinned host buffers, and independent chunks. Actual
overlap and speedup depend on the device and workload.
[NVIDIA CUDA C++ Best Practices Guide](https://docs.nvidia.com/cuda/archive/12.3.2/cuda-c-best-practices-guide/#asynchronous-and-overlapping-transfers-with-computation)


In [ ]:
from IPython.display import HTML, display

engines = ["Host → GPU copy engine", "SMs · kernel execution", "GPU → Host copy engine"]
streams = [("A", "#1d4ed8"), ("B", "#6d28d9"), ("C", "#047857")]
left, slot_width = 165, 92
parts = []
for stage, engine in enumerate(engines):
    y = 47 + stage * 43
    parts.append(
        f'<text x="12" y="{y + 21}">{engine}</text>'
        f'<rect x="{left}" y="{y}" width="460" height="32" rx="4" fill="#e2e8f0"/>'
    )
    for stream, (chunk, color) in enumerate(streams):
        x = left + (stage + stream) * slot_width
        parts.append(
            f'<rect x="{x + 2}" y="{y + 2}" width="88" height="28" rx="3" fill="{color}"/>'
            f'<text x="{x + 46}" y="{y + 21}" text-anchor="middle" fill="white">Chunk {chunk}</text>'
        )

display(HTML("""
<style>
.cuda-copy-pipeline { max-width: 650px; }
.cuda-copy-pipeline svg { display: block; width: 100%; height: auto; }
.cuda-copy-pipeline .engine-cursor { animation: cuda-engine-sweep 8s linear infinite; }
.cuda-copy-pipeline input:checked ~ svg .engine-cursor { animation-play-state: paused; }
@keyframes cuda-engine-sweep { to { transform: translateX(460px); } }
@media (prefers-reduced-motion: reduce) {
    .cuda-copy-pipeline .engine-cursor { display: none; }
}
</style>
<div class="cuda-copy-pipeline">
<input type="checkbox" id="cuda-copy-pipeline-pause">
<label for="cuda-copy-pipeline-pause">Pause cursor</label>
<svg viewBox="0 0 650 237" role="img" aria-labelledby="cuda-copy-pipeline-title">
  <title id="cuda-copy-pipeline-title">Copy engines and SMs work on different chunks simultaneously</title>
  <rect width="650" height="237" rx="8" fill="#f8fafc"/>
  <g font-family="system-ui, sans-serif" font-size="12" fill="#172033">
    <text x="12" y="25" font-size="15" font-weight="700">GPU hardware</text>
    <text x="395" y="25" text-anchor="middle" font-weight="700">All three active</text>
""" + "".join(parts) + """
    <rect x="349" y="38" width="92" height="130" rx="5" fill="none" stroke="#475569" stroke-dasharray="4 3"/>
    <path d="M165 187 H625 l-6 -4 m6 4 l-6 4" fill="none" stroke="#64748b"/>
    <text x="395" y="205" text-anchor="middle">Illustrative time</text>
    <text x="395" y="224" text-anchor="middle">5 slots with overlap · 9 slots if serialized</text>
  </g>
  <g class="engine-cursor" aria-hidden="true">
    <path d="M165 34 V175" stroke="white" stroke-width="4"/>
    <path d="M165 34 V175" stroke="#111827" stroke-width="2" stroke-dasharray="4 3"/>
  </g>
</svg>
</div>
"""))


## Main differences from the synchronous version


* Create and use a CUDA stream with `cudaStreamCreate()`.
* Use `cudaMemcpyAsync()` for asynchronous memory copies.
* Launch the kernel in the created stream.
* Synchronize the stream with `cudaStreamSynchronize()` to wait for completion.
* Cleanup the stream after use.



> **Important**
>
> `cudaMalloc` and `cudaFree` are still synchronous operations, meaning they block the host until the memory allocation or deallocation is complete.

```cpp
#include <stdio.h>
#include <cuda_runtime.h>

// Kernel 1: Adds 1 to each element
__global__ void kernel1(float* data, int n) 
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n)
        data[idx] += 1.0f;
}

// Kernel 2: Multiplies each element by 2
__global__ void kernel2(float* data, int n) 
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n)
        data[idx] *= 2.0f;
}

int main() 
{
    const int n1 = 512;
    const int n2 = 512;
    const int size1 = n1 * sizeof(float);
    const int size2 = n2 * sizeof(float);

    float *h_data1 = (float*)malloc(size1);
    float *h_data2 = (float*)malloc(size2);

    // Initialize host data
    for (int i = 0; i < n1; i++) h_data1[i] = i;
    for (int i = 0; i < n2; i++) h_data2[i] = i * 10;

    float *d_data1, *d_data2;
    cudaMalloc(&d_data1, size1);
    cudaMalloc(&d_data2, size2);

    cudaStream_t stream1, stream2;
    cudaStreamCreate(&stream1);
    cudaStreamCreate(&stream2);

    // Copy and launch kernel on stream1
    cudaMemcpyAsync(d_data1, h_data1, size1, cudaMemcpyHostToDevice, stream1);
    int threads = 256;
    int blocks1 = (n1 + threads - 1) / threads;
    kernel1<<<blocks1, threads, 0, stream1>>>(d_data1, n1);

    // Copy and launch kernel on stream2
    cudaMemcpyAsync(d_data2, h_data2, size2, cudaMemcpyHostToDevice, stream2);
    int blocks2 = (n2 + threads - 1) / threads;
    kernel2<<<blocks2, threads, 0, stream2>>>(d_data2, n2);

    // Copy results back asynchronously
    cudaMemcpyAsync(h_data1, d_data1, size1, cudaMemcpyDeviceToHost, stream1);
    cudaMemcpyAsync(h_data2, d_data2, size2, cudaMemcpyDeviceToHost, stream2);

    // Wait for streams to finish
    cudaStreamSynchronize(stream1);
    cudaStreamSynchronize(stream2);

    // Print some results
    printf("Stream 1 results:\n");
    for (int i = 0; i < 5; i++) printf("h_data1[%d] = %f\n", i, h_data1[i]);

    printf("Stream 2 results:\n");
    for (int i = 0; i < 5; i++) printf("h_data2[%d] = %f\n", i, h_data2[i]);

    // Cleanup
    cudaStreamDestroy(stream1);
    cudaStreamDestroy(stream2);
    cudaFree(d_data1);
    cudaFree(d_data2);
    free(h_data1);
    free(h_data2);

    return 0;
}
```

| API | Purpose |
| --- | --- |
| `cudaMemcpyAsync` | Starts a memory copy asynchronously and returns immediately. |
| `cudaStream_t` | Represents a CUDA stream (a queue of commands executed in order). |
| `cudaStreamCreate()` | Creates a new CUDA stream. |
| `cudaStreamSynchronize()` | Blocks host until all commands in the stream have completed. |
| `cudaStreamDestroy()` | Destroys the CUDA stream and frees associated resources. |

> **Key Points**
>
> 
>  1. Asynchronous CUDA APIs allow overlapping memory transfers and kernel execution.
>  1. CUDA streams enable concurrent execution of multiple tasks on the GPU.
>  1. Proper synchronization is crucial to ensure data integrity and correctness.
>  1. Cleanup of streams and device memory is necessary to avoid resource leaks.

## Exercise 06: Asynchronous CUDA Calls

**Time: 20 min**

Read the synchronous and asynchronous vector-add sources, then submit the job.
It runs both prebuilt programs in that order.
[Exercise instructions](../docs/source/tutorial/exercises/06-asynchronous.rst).

Run the setup cell first. Build the exercises once in
[02-gpu.ipynb](02-gpu.ipynb) before submitting jobs from this notebook.
After editing CUDA source, return to that notebook to rebuild.

Submission cells use the same `qsub` commands as the exercise documentation,
from `job-script`, and immediately show `qstat`. A submitted job may still be queued.
Use the monitoring cell to check again; rerunning a submission cell creates another job.
Read the PBS output and inspect generated files after the job finishes.

<a href="../exercises/06-asynchronous/vector-add.cu" target="_blank" rel="noopener noreferrer">Open vector-add.cu</a>

<a href="../exercises/06-asynchronous/vector-add-async.cu" target="_blank" rel="noopener noreferrer">Open vector-add-async.cu</a>

In [ ]:
# Submit a new job, then inspect the queue.
submitted_at = time.time()
submission = !cd "{job_dir}" && qsub 06-asynchronous.pbs
print("\n".join(submission))
!qstat -u "{username}"

# Keep the returned job ID so later cells read this run's PBS log.
job_id = next((line.strip() for line in submission
                 if re.fullmatch(r"\d+(?:\.[A-Za-z0-9_.-]+)?", line.strip())), None)
if job_id is None:
    raise RuntimeError("qsub did not return a job ID. Check its output above.")


### Check the queue again

Run this cell to monitor jobs without submitting another one. Once your job is no
longer listed, rerun its output/visualisation cells. Inspect the PBS log for errors.

In [ ]:
!qstat -u "{username}"


In [ ]:
!qstat -u "{username}"
show_job_output("06-asynchronous", job_id)


### Compare the results

| Version | Reported total time (ms) | Verification |
| --- | --- | --- |
| Synchronous | | |
| Asynchronous, four streams | | |

Read the timing boundaries before interpreting the difference. The synchronous
version includes per-iteration verification and printing; the asynchronous
version verifies after timing. Its host buffers use ordinary `malloc`, so transfer
and kernel overlap is not guaranteed. These reported times do not isolate the
benefit of streams. Repeat the same documented job to explore variability.